# 🔬 IMD Streaming Workshop - Solutions

Complete solutions for the workshop exercises with live visualizations.

---

## Exercise 1 Solution: Charge Density Analysis

Calculate the total charge within 8 Å of the protein at each timestep.

### 🧬 About Lysozyme's Charge

**Expected Result**: Lysozyme has a net charge of **+8e**, so we expect to see a **net negative charge** in the surrounding solvent due to:
- **Chloride ions (Cl⁻)** being attracted to the positively charged protein surface
- **Sodium ions (Na⁺)** being repelled from the protein
- **Water molecules** orienting with their negative oxygen atoms toward the protein

This creates an **ionic atmosphere** around the protein that screens its positive charge!

In [ ]:
import MDAnalysis as mda
import numpy as np
from graph_utils import live_plot

u = mda.Universe("sample_simulation/GROMACS/input/start.tpr", "imd://localhost:8889", buffer_size=100*1024**2)

nearby = u.select_atoms('not protein and around 8 protein', updating=True)
plot = live_plot("Charge Density within 8 Å", ylabel="Total Charge (e)")

try:
    for ts in u.trajectory:
        total_charge = np.sum(nearby.charges)
        plot['update'](ts.time, total_charge)
        print(f"Frame {ts.frame:4d} | Time: {ts.time:8.2f} ps | Net Charge: {total_charge} e")

except Exception as e:
    print(f"Error: {e}")
finally:
    u.trajectory.close()
    plot['close']()

---

## Exercise 2 Solution: Backbone RMSD

Calculate the backbone RMSD relative to the starting structure.

**What is backbone?** Only the main chain atoms (N, CA, C, O) that form the protein scaffold, excluding side chains.

**Why backbone RMSD?** More stable than all-atom RMSD since side chains are flexible. Captures overall structural changes.

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import numpy as np
from graph_utils import live_plot

u = mda.Universe("sample_simulation/GROMACS/input/start.gro", "imd://localhost:8889", buffer_size=100*1024**2)
# Have fixed axis for better visualization
# plot["ax"].set_xlim(0, 100)  # Time axis from 0 to 100 ps
# plot["ax"].set_ylim(13, 16)   # Rg axis from 13 to 16 Å

# Select backbone atoms (N, CA, C, O - the protein main chain)
backbone = u.select_atoms("protein and backbone")

# Create live plot
plot = live_plot(
    title="Backbone RMSD from Starting Structure",
    ylabel="RMSD (Å)",
    update_interval=1
)

try: 
    # Variable to store reference structure
    reference_positions = None

    for ts in u.trajectory:
        # Save reference positions from first frame
        if ts.frame == 0:
            reference_positions = backbone.positions.copy()
            # continue  # Skip RMSD calculation for frame 0 (RMSD = 0)
        
        # Calculate RMSD with alignment (superposition=True)
        rmsd_value = rms.rmsd(
            backbone.positions,
            reference_positions,
            superposition=True  # Align structures before calculating RMSD
        )
        
        plot['update'](ts.time, rmsd_value)

except Exception as e:
    print(f"\n\nError: {e}")
finally:
    u.trajectory.close()
    plot['close']()